# Carbon taxes

This page describes PolicyEngine UK's contributed carbon-tax model. It estimates a hypothetical household burden from consumption-related emissions, then allocates that burden between consumers and exposure to the corporate sector. It does not simulate an enacted carbon tax or a border adjustment.

## Domestic and overseas flows

Production location and consumption location are different questions. The model estimates the footprint of UK households' consumption without a country-of-production dimension; it does not measure each of the following flows separately.

| Where goods and services are produced | Where they are consumed | What the household model represents |
| --- | --- | --- |
| UK | UK households | Part of the household consumption footprint, without a separate domestic-production component. |
| Outside the UK | UK households | No separate imported-emissions variable. Any overseas production represented by the consumption-intensity inputs remains combined with domestic production; the formulas cannot identify its share or calculate a border adjustment. |
| UK | Outside the UK | No foreign-consumer population or separate export flow in the carbon-tax calculation. Stored UK production series are not its tax base. |
| Outside the UK | Outside the UK | No separate rest-of-world consumption or production flow in the UK household calculation. |

This distinction also applies to ownership. A household's exposure to corporate taxes does not identify where a company produces goods or where its customers live. The model does not calculate a separate burden for foreign owners or consumers, or separate employer and wage effects.

## From consumption to emissions

[`carbon_consumption`](https://github.com/PolicyEngine/policyengine-uk/blob/main/policyengine_uk/variables/household/consumption/carbon.py) is an annual household estimate in tonnes of CO2. It adds spending in twelve consumption categories, each multiplied by that category's carbon intensity. The intensity is aggregate category emissions divided by aggregate category spending; when that spending is nonpositive, the formula uses the stored fallback intensity.

The [carbon parameters](https://github.com/PolicyEngine/policyengine-uk/tree/main/policyengine_uk/parameters/household/consumption/carbon) also contain production and total-consumption series. The household footprint and carbon-tax formulas do not read those series to construct a territorial inventory or a production-to-consumption trade balance. Their presence does not establish separate coverage of all four flows above.

Consumption inputs, intensities and survey weights determine the resulting footprint. A sum over the simulated households is not by itself a validated UK national emissions total; that interpretation also requires suitable data and weights. This page describes the calculation, without validating a particular dataset's imputation or source-country coverage.

## How the burden is allocated

For a household with footprint `E_h`, the [`carbon_tax` formula](https://github.com/PolicyEngine/policyengine-uk/blob/main/policyengine_uk/variables/contrib/ubi_center/carbon_tax.py) is:

```text
E_total = sum(household_weight_h * E_h)
carbon_tax_h = c * r * E_h + (1 - c) * r * E_total * shareholding_h
```

Here `r` is the hypothetical price per tonne and `c` is the consumer-incidence fraction, both set by the [carbon-tax parameters](https://github.com/PolicyEngine/policyengine-uk/blob/main/policyengine_uk/parameters/gov/contrib/ubi_center/carbon_tax.yaml). The first term allocates the consumer share by the household's footprint. The second allocates the remaining burden using [`shareholding`](https://github.com/PolicyEngine/policyengine-uk/blob/main/policyengine_uk/variables/household/wealth/shareholding.py), the household's modelled corporate-sector wealth exposure. With a dataset and nonzero corporate wealth, that exposure is normalised by weighted household corporate wealth; otherwise the formula uses the national-balance-sheet denominator.

This is an allocation assumption, not a calculation of sector-specific demand elasticities, international ownership or the legal incidence of an import tax. Changing the incidence fraction changes who bears the modelled burden, without adding a country-of-production breakdown.

### Appendix: illustrative carbon intensities

The chart below uses fixed illustrative intensities in its code cell. It does not query the current carbon parameters and is not a new estimate or validation of domestic and overseas emissions.


In [1]:
import plotly.express as px
import pandas as pd

df = pd.DataFrame(
    {
        "Category": [
            "Alcohol and tobacco",
            "Clothing and footwear",
            "Communication",
            "Education",
            "Food",
            "Health",
            "Household furnishings",
            "Housing, water and electricity",
            "Miscellaneous",
            "Recreation",
            "Restaurants and hotels",
            "Transport",
        ],
        "Intensity": [
            7.7e-05,
            0.000262,
            0.000404,
            0.000241,
            0.000689,
            0.000656,
            0.000212,
            0.0012,
            0.000203,
            0.000185,
            0.000311,
            0.00107,
        ],
    }
)

df["Tonnes CO2 per £1,000"] = df.Intensity * 1000

px.bar(
    df.sort_values("Tonnes CO2 per £1,000", ascending=True),
    x="Tonnes CO2 per £1,000",
    y="Category",
    orientation="h",
).update_layout(
    title="Carbon intensity by consumption category",
    yaxis_title=None,
    width=800,
    height=600,
)